In [ ]:
import pandas as pd
import numpy as np 
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import seaborn as sns
import matplotlib.pyplot as plt


df_train = pd.read_csv('/kaggle/input/pruebas-saber/training_pruebas.csv', low_memory=False)

df_test = pd.read_csv('/kaggle/input/test-saberprubas/test_pruebas.csv', low_memory=False)

In [ ]:
# Eliminamos columnas que no tienen tanto peso para el estudio
columns_to_drop = [
    'ESTU_TIPODOCUMENTO', 'ESTU_TIPODOCUMENTOSB11', 'PERIODO',
    'INST_NOMBRE_INSTITUCION', 'ESTU_PRIVADO_LIBERTAD', 'ESTU_ESTADOINVESTIGACION', 'ESTU_ESTUDIANTE',
    'ESTU_NIVEL_PRGM_ACADEMICO', 'INST_CARACTER_ACADEMICO', 'ESTU_NACIONALIDAD', 'ESTU_TITULOOBTENIDOBACHILLER'
]
df_train.drop(columns=columns_to_drop, axis=1, inplace=True)
df_test.drop(columns=columns_to_drop, axis=1, inplace=True)

test_consecutivo = df_test['ESTU_CONSECUTIVO']
df_train.drop('ESTU_CONSECUTIVO', axis=1, inplace=True)

# Eliminamos columnas con demasiados valores nulos (umbral > 50%)
missing_ratio = df_train.isnull().mean()
columns_to_drop = missing_ratio[missing_ratio > 0.5].index.tolist()
df_train.drop(columns=columns_to_drop, axis=1, inplace=True)
df_test.drop(columns=columns_to_drop, axis=1, inplace=True)  # También en test


# Imputar valores numéricos en df_train 
numeric_cols = df_train.select_dtypes(include=['int64', 'float64']).columns
numeric_imputation = {}

for col in numeric_cols:
    mean = df_train[col].mean()
    median = df_train[col].median()
    
    # Decidir si usar la media o la mediana según la dispersión
    if abs(mean - median) / mean > 0.1:
        df_train[col] = df_train[col].fillna(median)
        numeric_imputation[col] = median
    else:
        df_train[col] = df_train[col].fillna(mean)
        numeric_imputation[col] = mean

# Imputar valores categóricos en df_train 
categorical_cols = df_train.select_dtypes(include=['object']).columns
categorical_imputation = {}

for col in categorical_cols:
    mode = df_train[col].mode()[0]
    df_train[col] = df_train[col].fillna(mode)
    categorical_imputation[col] = mode


# Imputar valores numéricos en df_test
for col, value in numeric_imputation.items():
    if col in df_test.columns:
        df_test[col] = df_test[col].fillna(value)

# Imputar valores categóricos en df_test
for col, value in categorical_imputation.items():
    if col in df_test.columns:
        df_test[col] = df_test[col].fillna(value)

#Crear columna de edad y eliminar 'ESTU_FECHANACIMIENTO'
def calcular_edad(fecha_nacimiento):
    fecha_nacimiento = pd.to_datetime(fecha_nacimiento, errors='coerce', dayfirst=True)
    hoy = pd.Timestamp('now')
    cumple_ya_ocurrido = (hoy.month > fecha_nacimiento.dt.month) | (
        (hoy.month == fecha_nacimiento.dt.month) & (hoy.day >= fecha_nacimiento.dt.day)
    )
    edad = hoy.year - fecha_nacimiento.dt.year - (~cumple_ya_ocurrido).astype(int)
    return edad

# Aplicar en df_train
df_train['ESTU_EDAD'] = calcular_edad(df_train['ESTU_FECHANACIMIENTO'])
df_train.drop('ESTU_FECHANACIMIENTO', axis=1, inplace=True)
median_age = df_train['ESTU_EDAD'].median()
df_train['ESTU_EDAD'].fillna(median_age, inplace=True)

# Aplicar en df_test
df_test['ESTU_EDAD'] = calcular_edad(df_test['ESTU_FECHANACIMIENTO'])
df_test.drop('ESTU_FECHANACIMIENTO', axis=1, inplace=True)
df_test['ESTU_EDAD'].fillna(median_age, inplace=True)

#Codificar variables binarias
binary_vars = [
    'FAMI_TIENECONSOLAVIDEOJUEGOS', 'FAMI_TIENEMOTOCICLETA', 'FAMI_TIENEAUTOMOVIL',
    'FAMI_TIENESERVICIOTV', 'FAMI_TIENEHORNOMICROOGAS', 'FAMI_TIENELAVADORA',
    'FAMI_TIENECOMPUTADOR', 'FAMI_TIENEINTERNET',
    'ESTU_PAGOMATRICULABECA', 'ESTU_PAGOMATRICULACREDITO',
    'ESTU_PAGOMATRICULAPADRES', 'ESTU_PAGOMATRICULAPROPIO',
    'ESTU_EXTERIOR', 'ESTU_GENERO'
]

binary_mapping = {'No': 0, 'Sí': 1, 'Si': 1, 'NO': 0, 'SI': 1, 'M': 0, 'F': 1, 'Masculino': 0, 'Femenino': 1}

#valores faltantes en estas variables
for var in binary_vars:
    if var in df_train.columns:
        df_train[var].fillna('No', inplace=True)
    if var in df_test.columns:
        df_test[var].fillna('No', inplace=True)

# Mapear las variables binarias
for var in binary_vars:
    if var in df_train.columns:
        df_train[var] = df_train[var].map(binary_mapping).astype(int)
    if var in df_test.columns:
        df_test[var] = df_test[var].map(binary_mapping).astype(int)

#Codificar variables ordinales

# 'FAMI_ESTRATOVIVIENDA'
estrato_order = ['Estrato 1', 'Estrato 2', 'Estrato 3', 'Estrato 4', 'Estrato 5', 'Estrato 6']
for df_ in [df_train, df_test]:
    if 'FAMI_ESTRATOVIVIENDA' in df_.columns:
        df_['FAMI_ESTRATOVIVIENDA'] = pd.Categorical(
            df_['FAMI_ESTRATOVIVIENDA'],
            categories=estrato_order,
            ordered=True
        ).codes

# 'ESTU_SEMESTRECURSA'
for df_ in [df_train, df_test]:
    if 'ESTU_SEMESTRECURSA' in df_.columns:
        df_['ESTU_SEMESTRECURSA'] = df_['ESTU_SEMESTRECURSA'].replace('12 o más', '12').astype(int)

# 'FAMI_CUANTOSCOMPARTEBAÑO'
orden_cuanto_bano = ['Ninguna', '1', '2', '3 o 4', '5 o 6', 'Más de 6']
for df_ in [df_train, df_test]:
    if 'FAMI_CUANTOSCOMPARTEBAÑO' in df_.columns:
        df_['FAMI_CUANTOSCOMPARTEBAÑO'] = pd.Categorical(
            df_['FAMI_CUANTOSCOMPARTEBAÑO'],
            categories=orden_cuanto_bano,
            ordered=True
        ).codes

# 'ESTU_VALORMATRICULAUNIVERSIDAD'
orden_matricula = [
    'No pagó matrícula', 'Menos de 500 mil', 'Entre 500 mil y menos de 1 millón',
    'Entre 1 millón y menos de 2.5 millones', 'Entre 2.5 millones y menos de 4 millones',
    'Entre 4 millones y menos de 5.5 millones', 'Entre 5.5 millones y menos de 7 millones',
    'Más de 7 millones'
]
for df_ in [df_train, df_test]:
    if 'ESTU_VALORMATRICULAUNIVERSIDAD' in df_.columns:
        df_['ESTU_VALORMATRICULAUNIVERSIDAD'] = pd.Categorical(
            df_['ESTU_VALORMATRICULAUNIVERSIDAD'],
            categories=orden_matricula,
            ordered=True
        ).codes

# 'FAMI_EDUCACIONMADRE' y 'FAMI_EDUCACIONPADRE'
orden_educacion = [
    'Ninguno', 'Primaria incompleta', 'Primaria completa', 'Secundaria incompleta',
    'Secundaria completa', 'Técnica o tecnológica incompleta', 'Técnica o tecnológica completa',
    'Educación profesional incompleta', 'Educación profesional completa', 'Postgrado'
]
for df_ in [df_train, df_test]:
    for col in ['FAMI_EDUCACIONMADRE', 'FAMI_EDUCACIONPADRE']:
        if col in df_.columns:
            df_[col] = pd.Categorical(
                df_[col],
                categories=orden_educacion,
                ordered=True
            ).codes

# Encoding para variables nominales
nominal_vars = ['ESTU_ESTADOCIVIL', 'ESTU_COMOCAPACITOEXAMENSB11']
df_train = pd.get_dummies(df_train, columns=nominal_vars, drop_first=True)
df_test = pd.get_dummies(df_test, columns=nominal_vars, drop_first=True)

if 'ESTU_CONSECUTIVO' not in df_test.columns:
    df_test['ESTU_CONSECUTIVO'] = test_consecutivo

In [ ]:
#Agrupar categorías poco frecuentes y eliminar variables problemáticas

def agrupar_categorias_poco_frecuentes(df, column, threshold=100):
    if column in df.columns:  # Verificar si la columna existe
        counts = df[column].value_counts()
        df[column] = df[column].apply(lambda x: x if counts[x] >= threshold else 'Otros')
    return df

# Agrupar categorías poco frecuentes en df_train
df_train = agrupar_categorias_poco_frecuentes(df_train, 'ESTU_DEPTO_RESIDE', threshold=100)
df_train = agrupar_categorias_poco_frecuentes(df_train, 'ESTU_DEPTO_PRESENTACION', threshold=100)

# Agrupar categorías poco frecuentes en df_test
df_test = agrupar_categorias_poco_frecuentes(df_test, 'ESTU_DEPTO_RESIDE', threshold=100)
df_test = agrupar_categorias_poco_frecuentes(df_test, 'ESTU_DEPTO_PRESENTACION', threshold=100)

# Aplicar One-Hot Encoding a las variables categóricas restantes
categorical_cols = [
    'ESTU_DEPTO_RESIDE', 'ESTU_MCPIO_RESIDE',
    'ESTU_AREARESIDE', 'FAMI_OCUPACIONPADRE', 'FAMI_OCUPACIONMADRE',
    'GRUPOREFERENCIA', 'ESTU_PRGM_ACADEMICO', 'ESTU_PRGM_MUNICIPIO',
    'ESTU_PRGM_DEPARTAMENTO', 'ESTU_METODO_PRGM', 'INST_ORIGEN',
    'ESTU_DEPTO_PRESENTACION'
]

# Aplicar Encoding a las columnas categóricas en ambos conjuntos de datos
df_train = pd.get_dummies(df_train, columns=categorical_cols, drop_first=True)
df_test = pd.get_dummies(df_test, columns=categorical_cols, drop_first=True)

# Alinear columnas entre df_train y df_test
df_train, df_test = df_train.align(df_test, join='left', axis=1, fill_value=0)

#Eliminamos variables con baja correlación
low_correlation_features = ['ESTU_COD_RESIDE_DEPTO', 'ESTU_INST_CODMUNICIPIO', 'ESTU_PRGM_CODMUNICIPIO',
                            'ESTU_GENERO', 'ESTU_PAGOMATRICULACREDITO', 'ESTU_COD_RESIDE_MCPIO', 
                            'FAMI_TIENEMOTOCICLETA', 'ESTU_COD_DEPTO_PRESENTACION', 
                            'ESTU_COD_MCPIO_PRESENTACION', 'ESTU_PAGOMATRICULAPROPIO']
df_train.drop(columns=low_correlation_features, inplace=True)
df_test.drop(columns=low_correlation_features, inplace=True)

# Codificar 'FAMI_TRABAJOLABORPADRE' y 'FAMI_TRABAJOLABORMADRE'
def agrupar_trabajo(trabajo):
    if 'auxiliar' in trabajo.lower() or 'técnico' in trabajo.lower():
        return 'Auxiliar/Técnico'
    elif 'profesional' in trabajo.lower():
        return 'Profesional'
    elif 'operario' in trabajo.lower() or 'conduce' in trabajo.lower():
        return 'Operario/Conductor'
    elif 'hogar' in trabajo.lower():
        return 'Hogar'
    elif 'pensionado' in trabajo.lower():
        return 'Pensionado'
    elif 'no aplica' in trabajo.lower():
        return 'No Aplica'
    else:
        return 'Otros'

df_train['FAMI_TRABAJOLABORPADRE'] = df_train['FAMI_TRABAJOLABORPADRE'].apply(agrupar_trabajo)
df_train['FAMI_TRABAJOLABORMADRE'] = df_train['FAMI_TRABAJOLABORMADRE'].apply(agrupar_trabajo)
df_train = pd.get_dummies(df_train, columns=['FAMI_TRABAJOLABORPADRE', 'FAMI_TRABAJOLABORMADRE'], drop_first=True)

df_test['FAMI_TRABAJOLABORPADRE'] = df_test['FAMI_TRABAJOLABORPADRE'].apply(agrupar_trabajo)
df_test['FAMI_TRABAJOLABORMADRE'] = df_test['FAMI_TRABAJOLABORMADRE'].apply(agrupar_trabajo)
df_test = pd.get_dummies(df_test, columns=['FAMI_TRABAJOLABORPADRE', 'FAMI_TRABAJOLABORMADRE'], drop_first=True)

# Codificar 'ESTU_HORASSEMANATRABAJA'
horas_order = ['0', 'Menos de 10 horas', 'Entre 11 y 20 horas', 'Entre 21 y 30 horas', 'Más de 30 horas']
df_train['ESTU_HORASSEMANATRABAJA'] = pd.Categorical(df_train['ESTU_HORASSEMANATRABAJA'], categories=horas_order, ordered=True).codes
df_test['ESTU_HORASSEMANATRABAJA'] = pd.Categorical(df_test['ESTU_HORASSEMANATRABAJA'], categories=horas_order, ordered=True).codes

# Codificar 'ESTU_VLRULTIMOSEMESCURSADO'
valor_semestre_order = ['No pago semestre', 'Menos de 500 mil pesos', 'Entre un millón y 3 millones de pesos',
                        'Entre 3 y 5 millones', 'Entre 5 y 7 millones', 'Más de 7 millones']
df_train['ESTU_VLRULTIMOSEMESCURSADO'] = pd.Categorical(df_train['ESTU_VLRULTIMOSEMESCURSADO'], categories=valor_semestre_order, ordered=True).codes
df_test['ESTU_VLRULTIMOSEMESCURSADO'] = pd.Categorical(df_test['ESTU_VLRULTIMOSEMESCURSADO'], categories=valor_semestre_order, ordered=True).codes

# Codificar 'ESTU_PRESENTACIONSABADO'
df_train.loc[:, 'ESTU_PRESENTACIONSABADO'] = df_train['ESTU_PRESENTACIONSABADO'].fillna('No')
df_test.loc[:, 'ESTU_PRESENTACIONSABADO'] = df_test['ESTU_PRESENTACIONSABADO'].fillna('No')
df_train['ESTU_PRESENTACIONSABADO'] = df_train['ESTU_PRESENTACIONSABADO'].map({'Si': 1, 'No': 0}).astype(int)
df_test['ESTU_PRESENTACIONSABADO'] = df_test['ESTU_PRESENTACIONSABADO'].map({'Si': 1, 'No': 0}).astype(int)

# Verificar si quedan columnas no numéricas
non_numeric_columns_train = df_train.select_dtypes(include=['object', 'category']).columns
non_numeric_columns_test = df_test.select_dtypes(include=['object', 'category']).columns

if len(non_numeric_columns_train) > 0:
    print(f"Columnas no numéricas en df_train: {non_numeric_columns_train}")

if len(non_numeric_columns_test) > 0:
    print(f"Columnas no numéricas en df_test: {non_numeric_columns_test}")

In [ ]:
# Seleccionar columnas numéricas
numeric_cols = df_train.select_dtypes(include=['int64', 'float64']).columns

# Calcular la matriz de correlación solo para las variables numéricas
correlation_matrix = df_train[numeric_cols].corr()

# Visualizar la correlación usando un heatmap para las principales 10 variables
plt.figure(figsize=(10, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)
plt.title('Matriz de Correlación entre las Variables Numéricas')
plt.show()

# Mostrar solo las correlaciones de 'PUNT_GLOBAL'
target_correlation = correlation_matrix['PUNT_GLOBAL'].sort_values(ascending=False)
print('Correlación con PUNT_GLOBAL:')
print(target_correlation)

# Paso 10: Eliminar variables con baja correlación
low_correlation_features = target_correlation[abs(target_correlation) < 0.05].index.tolist()
low_correlation_features = [col for col in low_correlation_features if col != 'PUNT_GLOBAL']

# Eliminar columnas con baja correlación del conjunto de entrenamiento y de prueba
df_train.drop(columns=low_correlation_features, inplace=True)
df_test.drop(columns=low_correlation_features, inplace=True)

Exploración de los datos:

calculamos la correlación entre las variables y PUNT_GLOBAL despues de tener de haber echo la limpieza de datos y haber generado los valores dummy. Esto nos ayuda a identificar qué variables son relevantes para el modelo de predicción y vemos como el nivel socioeconómico (ESTU_NSE_IES, ESTU_INSE_INDIVIDUAL) y el acceso a recursos en el hogar (FAMI_TIENECOMPUTADOR, FAMI_TIENEINTERNET) tienen una correlación positiva con el puntaje.

In [ ]:
def plot_scatter(df, variables, target):
    for var in variables:
        plt.figure(figsize=(8, 6))
        sns.scatterplot(x=df[var], y=df[target])
        plt.title(f'Relación entre {var} y {target}')
        plt.xlabel(var)
        plt.ylabel(target)
        plt.grid(True)
        plt.show()

high_corr_vars = ['ESTU_NSE_IES', 'ESTU_INSE_INDIVIDUAL', 'ESTU_NSE_INDIVIDUAL', 'FAMI_TIENEHORNOMICROOGAS']
plot_scatter(df_train, high_corr_vars, 'PUNT_GLOBAL')

Ninguna de las variables visualizadas en estos gráficos parece tener una relación muy fuerte o lineal con el puntaje global (PUNT_GLOBAL). si hay cierta correlación positiva en el caso del ESTU_INSE_INDIVIDUAL, en general, los puntajes se distribuyen ampliamente dentro de cada categoría de estrato socioeconómico. La dispersión sugiere que es probable que existan otros factores más importantes o combinaciones de variables que expliquen mejor las variaciones en los puntajes.

In [ ]:
def plot_boxplot(df, variables, target):
    for var in variables:
        plt.figure(figsize=(8, 6))
        sns.boxplot(x=df[var], y=df[target])
        plt.title(f'Boxplot de {target} por {var}')
        plt.xlabel(var)
        plt.ylabel(target)
        plt.grid(True)
        plt.show()

categorical_vars = ['FAMI_TIENEINTERNET', 'FAMI_TIENEAUTOMOVIL', 'FAMI_TIENECOMPUTADOR']
plot_boxplot(df_train, categorical_vars, 'PUNT_GLOBAL')

Factores relacionados con el acceso a tecnología: Variables como FAMI_TIENEINTERNET y FAMI_TIENECOMPUTADOR muestran que el acceso a tecnología está relacionado con mejores puntajes en las pruebas del saber, pero de igual manera que en caso anterior por la alta dispersion de puntajes se puede deber a que existe mas factores que afectan el rendiemeinto en la prueba.

In [ ]:
def plot_histograms(df, variables):
    for var in variables:
        plt.figure(figsize=(8, 6))
        sns.histplot(df[var], kde=True)
        plt.title(f'Histograma de {var}')
        plt.xlabel(var)
        plt.ylabel('Frecuencia')
        plt.grid(True)
        plt.show()

numeric_vars = ['PUNT_GLOBAL', 'ESTU_EDAD', 'ESTU_NSE_IES']
plot_histograms(df_train, numeric_vars)

La mayoría de los estudiantes tiene puntajes en un rango intermedio que estan alrededor de 100-160, lo cual podria deberse a que la mayoría logra un rendimiento medio en las pruebas del saber. Hay algunos casos extremos, pero son relativamente pocos.

La mayor parte de la población estudiantil que participa en las pruebas del saber se encuentra en el rango típico de la educación universitaria o preuniversitaria. Los estudiantes de mayor edad son casos atípicos en este conjunto de datos.

La mayor parte de las instituciones educativas parecen concentrarse en niveles socioeconómicos intermedios como estratos 2, 3 y 4, lo que refleja una cierta diversidad económica en las instituciones donde estudian los participantes de las pruebas del saber

In [ ]:
# Paso 10: Definir X e y antes de continuar
X = df_train.drop('PUNT_GLOBAL', axis=1)
y = df_train['PUNT_GLOBAL']

X_test = df_test.copy()

# Si todavía existen columnas categóricas, asegurarse de que estén codificadas como dummy variables
# Este paso debería eliminar cualquier problema con las variables categóricas
X = pd.get_dummies(X, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

# Alinear las columnas entre df_train y df_test
X, X_test = X.align(X_test, join='left', axis=1, fill_value=0)

# Dividir el conjunto de datos
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Escalar las variables numéricas
scaler = StandardScaler()
numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns


X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_val_scaled[numeric_cols] = scaler.transform(X_val[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

# Buscar el mejor valor de alpha con GridSearchCV
param_grid = {'alpha': [0.1, 1.0, 10, 100]}
grid_search = GridSearchCV(Ridge(), param_grid, cv=5)
grid_search.fit(X_train_scaled, y_train)
best_alpha = grid_search.best_params_['alpha']
print(f"Mejor valor de alpha encontrado: {best_alpha}")

# Entrenar el modelo con el mejor valor de alpha
model_best = Ridge(alpha=best_alpha)
model_best.fit(X_train_scaled, y_train)

# Predecir sobre el conjunto de validación y prueba
y_pred_val = model_best.predict(X_val_scaled)
y_test_pred = model_best.predict(X_test_scaled)

# Evaluar el modelo en el conjunto de validación
mse_val = mean_squared_error(y_val, y_pred_val)
rmse_val = np.sqrt(mse_val)
mae_val = mean_absolute_error(y_val, y_pred_val)
r2_val = r2_score(y_val, y_pred_val)

print('--- Métricas de Evaluación en el Conjunto de Validación ---')
print(f"Mean Squared Error (MSE): {mse_val:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse_val:.2f}")
print(f"Mean Absolute Error (MAE): {mae_val:.2f}")
print(f"Coeficiente de Determinación (R²): {r2_val:.4f}")

# Calcular el MAPE (Mean Absolute Percentage Error)
def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    non_zero_indices = y_true != 0
    y_true_non_zero = y_true[non_zero_indices]
    y_pred_non_zero = y_pred[non_zero_indices]
    return np.mean(np.abs((y_true_non_zero - y_pred_non_zero) / y_true_non_zero)) * 100

mape_val = mean_absolute_percentage_error(y_val, y_pred_val)
print(f"Mean Absolute Percentage Error (MAPE): {mape_val:.2f}%")

# Crear el archivo de salida con las predicciones
df_submission = pd.DataFrame({
    'ESTU_CONSECUTIVO': test_consecutivo,  # Asegúrate de que esta variable está correctamente guardada
    'PUNT_GLOBAL': y_test_pred  # Asegúrate de que y_test_pred existe
})

# Guardar el archivo en formato CSV
df_submission.to_csv('submission.csv', index=False)